LOB and Multi Agent 

In [1]:
import random
import numpy as np
import pandas as pd
import heapq
import itertools
import time
from abc import ABC, abstractmethod
import plotly.graph_objects as go

class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  # Max-Heap (negative prices)
        self.asks = []  # Min-Heap (positive prices)
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

    def get_snapshot_levels(self, depth=10):
        """
        Returns the top N levels of the book for the heatmap.
        Format: {'bids': {price: vol}, 'asks': {price: vol}}
        """
        snapshot = {'bids': {}, 'asks': {}}
        
        # Aggregate Bids
        temp_bids = self.bids.copy()
        count = 0
        while temp_bids and count < depth:
            order = heapq.heappop(temp_bids)
            price = -order[0]
            qty = order[2]
            snapshot['bids'][price] = snapshot['bids'].get(price, 0) + qty
            count += 1
            
        # Aggregate Asks
        temp_asks = self.asks.copy()
        count = 0
        while temp_asks and count < depth:
            order = heapq.heappop(temp_asks)
            price = order[0]
            qty = order[2]
            snapshot['asks'][price] = snapshot['asks'].get(price, 0) + qty
            count += 1
            
        return snapshot


class Agent(ABC):
    def __init__(self, agent_id):
        self.agent_id = agent_id
    
    @abstractmethod
    def get_action(self, market_snapshot):
        pass

class NoiseTrader(Agent):
    """Adds random liquidity to the book."""
    def get_action(self, snapshot):
        if random.random() > 0.3: return None
        
        mid = snapshot['mid_price']
        side = 'buy' if random.random() > 0.5 else 'sell'
        
        # Place limit order near mid price
        offset = random.uniform(0.1, 2.0)
        price = round(mid - offset, 2) if side == 'buy' else round(mid + offset, 2)
        qty = random.randint(1, 10)
        
        return {'side': side, 'price': price, 'qty': qty, 'type': 'limit'}

class MarketMaker(Agent):
    """Adds stable liquidity around the spread."""
    def get_action(self, snapshot):
        mid = snapshot['mid_price']
        spread = 0.5
        
        return [
            {'side': 'buy', 'price': round(mid - spread, 2), 'qty': 5, 'type': 'limit'},
            {'side': 'sell', 'price': round(mid + spread, 2), 'qty': 5, 'type': 'limit'}
        ]

class RLAgentWrapper(Agent):
    """
    In a full production run, you would load 'models/ppo_v1_day4' here.
    For this standalone viz script, we simulate an 'Active' agent 
    similar to your passing Day 5 run.
    """
    def get_action(self, snapshot):
        # Simulated "Hyper-Active" Agent Logic
        if random.random() > 0.2: return None 
        
        side = 'buy' if random.random() > 0.5 else 'sell'
        # Market orders to take liquidity
        return {'side': side, 'price': None, 'qty': 1, 'type': 'market'} 

# 3. SIMULATION LOOP & DATA LOGGING

def run_simulation(steps=1000):
    print(f"--- Running Multi-Agent Simulation ({steps} Steps) ---")
    
    engine = AdvancedOrderBook()
    
    # Initialize Agents
    agents = []
    # 1 RL Agent
    agents.append(RLAgentWrapper(0))
    # 50 Noise Traders (The Ocean)
    for i in range(50): agents.append(NoiseTrader(i+1))
    # 5 Market Makers (The Stabilizers)
    for i in range(5): agents.append(MarketMaker(i+51))
    
    # Init Engine with a tight spread
    engine.submit_order('buy', 10, 99.0, 'limit')
    engine.submit_order('sell', 10, 101.0, 'limit')
    
    # Data Storage
    heatmap_data = [] # List of {'time': t, 'price': p, 'volume': v}
    trade_prices = [] # List of {'time': t, 'price': p}
    
    current_time = 0
    
    for step in range(steps):
        current_time += 1
        
        # 1. Market Snapshot
        best_bid = -engine.bids[0][0] if engine.bids else 100.0
        best_ask = engine.asks[0][0] if engine.asks else 100.0
        mid = (best_bid + best_ask) / 2
        
        snapshot = {'mid_price': mid}
        
        # 2. Log LOB State for Heatmap (The "Microscope")
        levels = engine.get_snapshot_levels(depth=20)
        
        # Log Bids
        for price, vol in levels['bids'].items():
            # Filter distinct outliers to keep chart readable
            if 90 < price < 110: 
                heatmap_data.append({'time': current_time, 'price': price, 'volume': vol, 'side': 'bid'})
                
        # Log Asks
        for price, vol in levels['asks'].items():
            if 90 < price < 110:
                heatmap_data.append({'time': current_time, 'price': price, 'volume': vol, 'side': 'ask'})
        
        # 3. Agents Act
        random.shuffle(agents)
        for agent in agents:
            actions = agent.get_action(snapshot)
            
            if isinstance(actions, list):
                for order in actions:
                    engine.submit_order(order['side'], order['qty'], order['price'], order['type'])
            elif actions:
                engine.submit_order(actions['side'], actions['qty'], actions['price'], actions['type'])
                
        # 4. Log Trades
        while engine.trades:
            trade = engine.trades.pop(0)
            trade_prices.append({'time': current_time, 'price': trade['price']})
            
    return pd.DataFrame(heatmap_data), pd.DataFrame(trade_prices)

# 4. VISUALIZATION (Plotly Heatmap)

def generate_heatmap(df_lob, df_trades):
    print("Generating Limit Order Book Heatmap...")
    
    if df_lob.empty:
        print("Error: No market data generated.")
        return

    # Pivot data: Index=Price, Columns=Time, Values=Volume
    # Round prices to 0.1 to bucket liquidity together
    df_lob['price_bucket'] = df_lob['price'].round(1)
    pivot_table = df_lob.pivot_table(index='price_bucket', columns='time', values='volume', aggfunc='sum').fillna(0)
    
    # Create Figure
    fig = go.Figure()

    # 1. Heatmap (The Liquidity)
    fig.add_trace(go.Heatmap(
        z=pivot_table.values,
        x=pivot_table.columns,
        y=pivot_table.index,
        colorscale='Viridis',
        showscale=True,
        name='Liquidity Depth'
    ))

    # 2. Trade Path (The Executions)
    if not df_trades.empty:
        fig.add_trace(go.Scatter(
            x=df_trades['time'],
            y=df_trades['price'],
            mode='markers',
            marker=dict(color='red', size=4),
            name='Executions'
        ))

    fig.update_layout(
        title="Limit Order Book Heatmap (Market Microstructure)",
        xaxis_title="Simulation Steps",
        yaxis_title="Price Level",
        template="plotly_dark",
        height=800
    )
    
    # Save as HTML
    output_file = "day6_market_heatmap.html"
    fig.write_html(output_file)
    print(f"Done! Open '{output_file}' in your web browser.")

if __name__ == "__main__":
    # Run slightly longer to generate a good looking map
    df_lob, df_trades = run_simulation(steps=2000)
    generate_heatmap(df_lob, df_trades)

--- Running Multi-Agent Simulation (2000 Steps) ---
Generating Limit Order Book Heatmap...
Done! Open 'day6_market_heatmap.html' in your web browser.
